In [ ]:
#https://www.kaggle.com/code/harshjain123/bert-for-everyone-tutorial-implementation
#https://medium.com/vmacwrites/pytorch-jupyter-notebook-modulenotfounderror-no-module-named-torch-e0f16dae1bdf
#%pip uninstall torch --y
#%pip install torch==1.5.0
#%pip install scikit-learn
#%pip install torch
%pip install transformers

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import transformers
from transformers import AutoModel, BertTokenizerFast, AutoTokenizer
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW

# specify GPU
device = torch.device("cuda")

In [ ]:
df = pd.read_csv("subset_dataset.csv")
df.rename(columns={'transcript': 'text'}, inplace=True)
df['label'] = df['label'].astype(int)
df.head(10)

In [ ]:
labels = df['label'].tolist()
texts = df['text'].tolist()

In [ ]:
# check class distribution
df['label'].value_counts(normalize = True)

In [ ]:
# get length of all the messages in the train set
seq_len = [len(i.split()) for i in df["text"]]

pd.Series(seq_len).hist(bins = 30)

In [ ]:
# split train dataset into train, validation and test sets
train_text, temp_text, train_labels, temp_labels = train_test_split(df['text'], df['label'], 
                                                                    random_state=42, 
                                                                    test_size=0.3, 
                                                                    stratify=df['label'])

val_text, test_text, val_labels, test_labels = train_test_split(temp_text, temp_labels, 
                                                                random_state=42, 
                                                                test_size=0.5, 
                                                                stratify=temp_labels)

In [ ]:
train_text

In [ ]:
# import BERT-base pretrained model
bert = AutoModel.from_pretrained('bert-base-uncased')

# Load the BERT tokenizer
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

In [ ]:
#tokenizer = AutoTokenizer.from_pretrained('albert/albert-base-v2')
#bert = AutoModel.from_pretrained("albert/albert-base-v2")

#tokenizer = AutoTokenizer.from_pretrained('FacebookAI/roberta-base')
#bert = AutoModel.from_pretrained("FacebookAI/roberta-base")

#tokenizer = AutoTokenizer.from_pretrained('vinai/bertweet-base')
#bert = AutoModel.from_pretrained("vinai/bertweet-base")

In [ ]:
#change from 55 to 120
ml = 120
# tokenize and encode sequences in the training set
tokens_train = tokenizer(
    train_text.tolist(),
    max_length = ml,
    padding=True,
    truncation=True
)

# tokenize and encode sequences in the validation set
tokens_val = tokenizer(
    val_text.tolist(),
    max_length = ml,
    padding=True,
    truncation=True
)

# tokenize and encode sequences in the test set
tokens_test = tokenizer(
    test_text.tolist(),
    max_length = ml,
    padding=True,
    truncation=True
)

In [ ]:
## convert lists to tensors

train_seq = torch.tensor(tokens_train['input_ids'])
train_mask = torch.tensor(tokens_train['attention_mask'])
train_y = torch.tensor(train_labels.tolist())

val_seq = torch.tensor(tokens_val['input_ids'])
val_mask = torch.tensor(tokens_val['attention_mask'])
val_y = torch.tensor(val_labels.tolist())

test_seq = torch.tensor(tokens_test['input_ids'])
test_mask = torch.tensor(tokens_test['attention_mask'])
test_y = torch.tensor(test_labels.tolist())

In [ ]:
train_seq.size()

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

#define a batch size
batch_size = 4

# wrap tensors
train_data = TensorDataset(train_seq, train_mask, train_y)

# sampler for sampling the data during training
train_sampler = RandomSampler(train_data)

# dataLoader for train set
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

# wrap tensors
val_data = TensorDataset(val_seq, val_mask, val_y)

# sampler for sampling the data during training
val_sampler = SequentialSampler(val_data)

# dataLoader for validation set
val_dataloader = DataLoader(val_data, sampler = val_sampler, batch_size=batch_size)

In [ ]:
# freeze all the parameters
for param in bert.parameters():
    param.requires_grad = False
    
class BERT_Arch(nn.Module):

    def __init__(self, bert):
        super(BERT_Arch, self).__init__()
        
        self.bert = bert 
                
        # dropout layer
        self.dropout = nn.Dropout(0.1)
      
        # relu activation function
        self.relu =  nn.ReLU()
        
        # dense layer 1
        self.fc1 = nn.Linear(768,512)
      
        # dense layer 2 (Output layer)
        self.fc2 = nn.Linear(512,2)

        #softmax activation function
        self.softmax = nn.LogSoftmax(dim=1)

    #define the forward pass
    def forward(self, sent_id, mask):
        
        #pass the inputs to the model  
        _, cls_hs = self.bert(sent_id, attention_mask=mask, return_dict=False)
      
        x = self.fc1(cls_hs)

        x = self.relu(x)

        x = self.dropout(x)

        # output layer
        x = self.fc2(x)
      
        # apply softmax activation
        x = self.softmax(x)

        return x

In [ ]:
# function to train the model
def train():
    
    model.train()
    total_loss, total_accuracy = 0, 0
  
    # empty list to save model predictions
    total_preds=[]
  
    # iterate over batches
    for step,batch in enumerate(train_dataloader):
        
                # progress update after every 50 batches.
        if step % 50 == 0 and not step == 0:
            print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(train_dataloader)))
        
        # push the batch to gpu
        batch = [r.to(device) for r in batch]
 
        sent_id, mask, labels = batch
        
        # clear previously calculated gradients 
        model.zero_grad()        
        
         # get model predictions for the current batch
        preds = model(sent_id, mask)

        # compute the loss between actual and predicted values
        loss = cross_entropy(preds, labels)

        # add on to the total loss
        total_loss = total_loss + loss.item()

        # backward pass to calculate the gradients
        loss.backward()

         # clip the the gradients to 1.0. It helps in preventing the exploding gradient problem
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # update parameters
        optimizer.step()

        # model predictions are stored on GPU. So, push it to CPU
        preds=preds.detach().cpu().numpy()

    # append the model predictions
    total_preds.append(preds)

    # compute the training loss of the epoch
    avg_loss = total_loss / len(train_dataloader)
  
      # predictions are in the form of (no. of batches, size of batch, no. of classes).
      # reshape the predictions in form of (number of samples, no. of classes)
    total_preds  = np.concatenate(total_preds, axis=0)

    #returns the loss and predictions
    return avg_loss, total_preds

In [ ]:
# function for evaluating the model
def evaluate():
    
    print("\nEvaluating...")
  
    # deactivate dropout layers
    model.eval()

    total_loss, total_accuracy = 0, 0
    
    # empty list to save the model predictions
    total_preds = []
    
    # iterate over batches
    for step,batch in enumerate(val_dataloader):
        
        # Progress update every 50 batches.
        if step % 50 == 0 and not step == 0:
            
            # Calculate elapsed time in minutes.
            elapsed = format_time(time.time() - t0)
            
            # Report progress.
            print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(val_dataloader)))
            
            
            # push the batch to gpu
        batch = [t.to(device) for t in batch]

        sent_id, mask, labels = batch

        # deactivate autograd
        with torch.no_grad():
            
            # model predictions
            preds = model(sent_id, mask)

            # compute the validation loss between actual and predicted values
            loss = cross_entropy(preds,labels)
            
            total_loss = total_loss + loss.item()

            preds = preds.detach().cpu().numpy()

            total_preds.append(preds)

    # compute the validation loss of the epoch
    avg_loss = total_loss / len(val_dataloader) 

    # reshape the predictions in form of (number of samples, no. of classes)
    total_preds  = np.concatenate(total_preds, axis=0)
    
    return avg_loss, total_preds

In [ ]:


blist = []
slist = []
splist = []
alist = []
runs = 5

for i in range(0,runs):

    # pass the pre-trained BERT to our define architecture
    model = BERT_Arch(bert)

    # push the model to GPU
    model = model.to(device)

    # define the optimizer
    optimizer = AdamW(model.parameters(),lr = 1e-6) 

    #compute the class weights
    class_weights = compute_class_weight('balanced', classes = np.unique(train_labels), y = train_labels)

    print("Class Weights:",class_weights)

    # converting list of class weights to a tensor
    weights= torch.tensor(class_weights,dtype=torch.float)

    # push to GPU
    weights = weights.to(device)

    # define the loss function
    cross_entropy  = nn.NLLLoss(weight=weights) 

    # number of training epochs
    epochs = 15

    # set initial loss to infinite
    best_valid_loss = float('inf')

    # empty lists to store training and validation loss of each epoch
    train_losses=[]
    valid_losses=[]

    #for each epoch
    for epoch in range(epochs):

        print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))

         #train model
        train_loss, _ = train()

        #evaluate model
        valid_loss, _ = evaluate()

        #save the best model
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), 'saved_weights.pt')

            # append training and validation loss
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        print(f'\nTraining Loss: {train_loss:.3f}')
        print(f'Validation Loss: {valid_loss:.3f}')

    #load weights of best model
    path = 'saved_weights.pt'
    model.load_state_dict(torch.load(path))

    # get predictions for test data
    with torch.no_grad():
        preds = model(test_seq.to(device), test_mask.to(device))
        preds = preds.detach().cpu().numpy()

    preds = np.argmax(preds, axis = 1)

    tn, fp, fn, tp = confusion_matrix(test_y, preds).ravel()
    sens = tp/(tp+fn)
    spec = tn/(tn+fp)
    ba = (sens+spec)/2
    acc = (tp+tn)/(tn+fp+fn+tp)
    #print("ba =", round(ba, 3), "sens =", round(sens, 3), "spec =", round(spec,3))

    blist.append(round(ba, 3))
    slist.append(round(sens, 3))
    splist.append(round(spec, 3))
    alist.append(round(acc, 3))

In [ ]:
print(blist)
print("ba =", sum(blist)/runs)
print("sens =", sum(slist)/runs)
print("spec =", sum(splist)/runs)
print("acc =", sum(alist)/runs)
print(alist)

In [ ]:
# code with learning rate = 5e-6

In [ ]:


blist = []
slist = []
splist = []
alist = []
runs = 5

for i in range(0,runs):

    # pass the pre-trained BERT to our define architecture
    model = BERT_Arch(bert)

    # push the model to GPU
    model = model.to(device)

    # define the optimizer
    # new learning rate for adam optimizer
    optimizer = AdamW(model.parameters(),lr = 5e-6) 

    #compute the class weights
    class_weights = compute_class_weight('balanced', classes = np.unique(train_labels), y = train_labels)

    print("Class Weights:",class_weights)

    # converting list of class weights to a tensor
    weights= torch.tensor(class_weights,dtype=torch.float)

    # push to GPU
    weights = weights.to(device)

    # define the loss function
    cross_entropy  = nn.NLLLoss(weight=weights) 

    # number of training epochs
    epochs = 15

    # set initial loss to infinite
    best_valid_loss = float('inf')

    # empty lists to store training and validation loss of each epoch
    train_losses=[]
    valid_losses=[]

    #for each epoch
    for epoch in range(epochs):

        print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))

         #train model
        train_loss, _ = train()

        #evaluate model
        valid_loss, _ = evaluate()

        #save the best model
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), 'saved_weights.pt')

            # append training and validation loss
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)

        print(f'\nTraining Loss: {train_loss:.3f}')
        print(f'Validation Loss: {valid_loss:.3f}')

    #load weights of best model
    path = 'saved_weights.pt'
    model.load_state_dict(torch.load(path))

    # get predictions for test data
    with torch.no_grad():
        preds = model(test_seq.to(device), test_mask.to(device))
        preds = preds.detach().cpu().numpy()

    preds = np.argmax(preds, axis = 1)

    tn, fp, fn, tp = confusion_matrix(test_y, preds).ravel()
    sens = tp/(tp+fn)
    spec = tn/(tn+fp)
    ba = (sens+spec)/2
    acc = (tp+tn)/(tn+fp+fn+tp)
    #print("ba =", round(ba, 3), "sens =", round(sens, 3), "spec =", round(spec,3))

    blist.append(round(ba, 3))
    slist.append(round(sens, 3))
    splist.append(round(spec, 3))
    alist.append(round(acc, 3))

In [ ]:
print(blist)
print("ba =", sum(blist)/runs)
print("sens =", sum(slist)/runs)
print("spec =", sum(splist)/runs)
print("acc =", sum(alist)/runs)
print(alist)

1. Report on average and standard deviation
For Learning Rate 1e-6:

Balanced Accuracies for 5 Runs:

Run 1: 0.467

Run 2: 0.500

Run 3: 0.500

Run 4: 0.500

Run 5: 0.433

Average Balanced Accuracy: 0.480

Standard Deviation: 0.030 (using sample standard deviation)

For Learning Rate 5e-6:

Balanced Accuracies for 5 Runs:

Run 1: 0.400

Run 2: 0.500

Run 3: 0.500

Run 4: 0.433

Run 5: 0.438

Average Balanced Accuracy: 0.454

Standard Deviation: 0.044 (using sample standard deviation)




2. Compare the predictiveness and stability

Neither model was predictive because they both basically failed to learn how to classify the minority class. The 1e-6 learning rate model achieved an average balanced accuracy of 0.480 with a sensitivity of 0.0, meaning it didn't correctly predict a single true positive across the runs. The 5e-6 learning rate was even less predictive, dropping to an average balanced accuracy of 0.454.

In terms of stability, the 1e-6 model was slightly more stable with a standard deviation of 0.030 compared to the 5e-6 model's 0.044. However, this stability just means it consistently failed. When examining the class slides from Module 9, it looks like BERT suffered from a 'bad' random seed and got stuck in a local minimum, which happens frequently with small datasets. Because of this, it mostly just guessed the majority class and completely missed the nuances of the data.

3. Compare BERT classifier performance with traditional ML models

Honestly, the traditional machine learning models we used in the P2 N2 notebook performed much better on this dataset than BERT did. For example, my Logistic Regression model hit a score of 0.63 and was actually able to catch some subtle hints in the text. BERT, on the other hand, completely failed to learn, resulting in balanced accuracies of 0.48 and 0.45.

BERT is supposed to be more advanced since it looks at the whole context of a sentence instead of just word counts. However, because our dataset is pretty small, BERT just got stuck predicting the majority label (as shown by its 0.0 sensitivity score). The simpler traditional models didn't get stuck like this, so they gave us much better balanced accuracy scores. Plus, they trained a lot faster than BERT.

4. Compare BERT performance with group members

When comparing my BERT model's performance to the rest of the group, it is clear that we all faced similar challenges in getting the model to consistently learn. I ran my BERT model using 15 epochs and a learning rate of 5e-6, which resulted in an average balanced accuracy of 0.4542 and a best individual balanced accuracy of 0.5. My model really struggled to pick up on the minority class and largely got stuck predicting the majority label, which dragged down the average. Grant also tested a learning rate of 5e-6, but his results were much more volatile. While his average balanced accuracy at that learning rate was slightly lower than mine at 0.436, he managed to achieve the highest single run of the group with a best balanced accuracy of 0.636. Grant also tested a lower 1e-6 learning rate, which yielded a more stable average of 0.499 and a best score of 0.529.

Jackson and Simon focused their efforts on lower learning rates to see if smaller steps would prevent the model from getting stuck in a local minimum. Jackson reduced his training from 10 to 5 epochs and found that a 1e-6 learning rate gave him an average balanced accuracy of 0.439 and a best score of 0.528. Interestingly, dropping his learning rate even further to 1e-7 improved his overall average accuracy to 0.483. Simon, on the other hand, had the most consistent results among all of us. Using a 1e-6 learning rate, his average balanced accuracy was 0.4596 with a best score of 0.5. When he slightly increased his learning rate to 2e-6, his model stabilized completely, resulting in both an average and best balanced accuracy of exactly 0.5.

Overall, almost everyone's average balanced accuracies hovered in the 0.43 to 0.50 range, which confirms my observation that BERT had a really hard time analyzing this specific dataset. Aside from Grant's one major spike to 0.636, our collective results show that simply tuning the learning rate between 1e-7 and 5e-6 was not enough to consistently overcome the model's tendency to just predict the majority class.